In [2]:
import numpy as np
from molgrid.molecule import Atom
from molgrid.atomicgrid import AtomicGrid

atom = Atom('O', [0.0, 0.0, 0.0])  # Oxygen at origin
    
# Create atomic grid
self_grid = AtomicGrid(atom, nshells=75, nangpts=302)

In [3]:
# Get radial grid
rid, rw =  self_grid.radial_coords, self_grid.radial_weights
print(f"Radial points: {len(rid)}")

 
# Get angular grid
ang, aw = self_grid.angular_coords, self_grid.angular_weights
print(f"Angular points: {len(ang)}")

# Get full grid
coords, weights = self_grid.coords, self_grid.weights
print(f"Total grid points: {len(coords)}")

Radial points: 75
Angular points: 302
Total grid points: 22650


In [10]:
import numpy as np
from pyscf import gto, dft
from pyscf.dft import numint

mol = gto.M(
    verbose = 0,
    atom = '''
    O    0    0.       0.
    ''',
    basis = '6-31g')

mf = dft.RKS(mol)
mf.kernel()


# Build the grid
pyscf_grids = dft.Grids(mol)
pyscf_grids.atom_grid = (75, 302)

pyscf_grids.prune = False
pyscf_grids.build()
print(f"Grid points: {len(pyscf_grids.coords)}, 75*302={75*302}")


# Get the density matrix
dm = mf.make_rdm1()


pyscf_ao = numint.eval_ao(mol, pyscf_grids.coords, deriv=0)
# The first row of rho is electron density, the rest three rows are electron
# density gradients which are needed for GGA functional
pyscf_rho = numint.eval_rho(mol, pyscf_ao, dm, xctype='LDA')
pyscf_ne = np.sum(pyscf_grids.weights * pyscf_rho)

print(f'number of electrons: {pyscf_ne}')

Grid points: 22656, 75*302=22650
number of electrons: 8.000000000000812


In [7]:
pyscf_grids.__dict__

{'mol': <pyscf.gto.mole.Mole at 0x117d5c830>,
 'stdout': <ipykernel.iostream.OutStream at 0x10e92b970>,
 'verbose': 0,
 'symmetry': False,
 'coords': array([[-4.34610632e+00, -4.34610632e+00, -4.34610632e+00],
        [-4.70569598e+00, -4.70569598e+00, -4.70569598e+00],
        [-5.12894526e+00, -5.12894526e+00, -5.12894526e+00],
        ...,
        [ 1.00000000e-04,  1.00000000e-04,  1.00000000e-04],
        [ 1.00000000e-04,  1.00000000e-04,  1.00000000e-04],
        [ 1.00000000e-04,  1.00000000e-04,  1.00000000e-04]],
       shape=(22656, 3)),
 'weights': array([1.48282656, 2.01854728, 2.86345725, ..., 0.        , 0.        ,
        0.        ], shape=(22656,)),
 'non0tab': None,
 'screen_index': None,
 'atm_idx': array([ 0,  0,  0, ..., -1, -1, -1], shape=(22656,), dtype=int32),
 'quadrature_weights': array([1.48282656, 2.01854728, 2.86345725, ..., 0.        , 0.        ,
        0.        ], shape=(22656,)),
 'atom_grid': (75, 302),
 'prune': False}

In [6]:
self_ao = numint.eval_ao(mol, self_grid.coords, deriv=0)
self_rho = numint.eval_rho(mol, self_ao, dm, xctype='LDA')
ne_self = np.sum(self_grid.weights * self_rho)

print(f'number of electrons: {ne_self}')

number of electrons: 7.999999999580259
